# Κεφάλαιο 7: Δημιουργία Εφαρμογών Συνομιλίας
## Γρήγορη Εκκίνηση API για τα Microsoft Foundry Models

Αυτό το σημειωματάριο προσαρμόστηκε από το [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) που περιλαμβάνει σημειωματάρια με πρόσβαση σε υπηρεσίες [Azure OpenAI](notebook-azure-openai.ipynb).

> **Σημείωση:** Το GitHub Models αποσύρεται στο τέλος Ιουλίου 2026. Αυτό το σημειωματάριο στοχεύει πλέον στα [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), που προσφέρουν τον ίδιο κατάλογο μοντέλων για δωρεάν δοκιμή και την εμπειρία του Azure AI Inference SDK.


# Επισκόπηση  
"Τα μεγάλα γλωσσικά μοντέλα είναι συναρτήσεις που αντιστοιχίζουν κείμενο σε κείμενο. Δοθείσας μιας εισερχόμενης ακολουθίας κειμένου, ένα μεγάλο γλωσσικό μοντέλο προσπαθεί να προβλέψει το κείμενο που θα ακολουθήσει"(1). Αυτό το σημειωματάριο "γρήγορης εκκίνησης" θα παρουσιάσει στους χρήστες τις βασικές έννοιες των LLM, τις βασικές απαιτήσεις πακέτων για να ξεκινήσουν με το AML, μια απλή εισαγωγή στο σχεδιασμό prompt, και μερικά σύντομα παραδείγματα διαφόρων περιπτώσεων χρήσης. 


## Περιεχόμενα  

[Επισκόπηση](#overview)  
[Πώς να χρησιμοποιήσετε την Υπηρεσία OpenAI](#how-to-use-openai-service)  
[1. Δημιουργία της Υπηρεσίας OpenAI σας](#1.-creating-your-openai-service)  
[2. Εγκατάσταση](#2.-installation)    
[3. Διαπιστευτήρια](#3.-credentials)  

[Περιπτώσεις Χρήσης](#use-cases)    
[1. Περίληψη Κειμένου](#1.-summarize-text)  
[2. Ταξινόμηση Κειμένου](#2.-classify-text)  
[3. Δημιουργία Νέων Ονομάτων Προϊόντων](#3.-generate-new-product-names)  
[4. Εκπαίδευση Ταξινομητή](#4.fine-tune-a-classifier)  

[Αναφορές](#references)


### Δημιουργήστε το πρώτο σας prompt  
Αυτή η σύντομη άσκηση θα σας δώσει μια βασική εισαγωγή για την υποβολή prompts σε ένα μοντέλο στα Microsoft Foundry Models για μια απλή εργασία "περίληψη".


**Βήματα**:  
1. Εγκαταστήστε τη βιβλιοθήκη `azure-ai-inference` στο περιβάλλον Python σας, αν δεν το έχετε ήδη κάνει.  
2. Φορτώστε τις βασικές βοηθητικές βιβλιοθήκες και ρυθμίστε τα διαπιστευτήρια Microsoft Foundry Models.  
3. Επιλέξτε ένα μοντέλο για την εργασία σας  
4. Δημιουργήστε ένα απλό prompt για το μοντέλο  
5. Υποβάλετε το αίτημά σας στο API του μοντέλου!


### 1. Εγκαταστήστε το `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Εισαγωγή βοηθητικών βιβλιοθηκών και δημιουργία διαπιστευτηρίων


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Εύρεση του κατάλληλου μοντέλου  
Τα μοντέλα όπως το GPT-4o και το GPT-4o mini μπορούν να κατανοούν και να παράγουν φυσική γλώσσα και είναι διαθέσιμα στον κατάλογο Microsoft Foundry Models μαζί με μοντέλα από τις Meta, Mistral, Cohere και Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Σχεδιασμός Ερωτημάτων  

"Η μαγεία των μεγάλων γλωσσικών μοντέλων είναι ότι με το να εκπαιδεύονται για να ελαχιστοποιούν αυτό το σφάλμα πρόβλεψης σε τεράστιες ποσότητες κειμένου, τα μοντέλα καταλήγουν να μαθαίνουν έννοιες χρήσιμες για αυτές τις προβλέψεις. Για παράδειγμα, μαθαίνουν έννοιες όπως"(1):

* πώς να γράφουν ορθογραφικά
* πώς λειτουργεί η γραμματική
* πώς να παραφράζουν
* πώς να απαντούν σε ερωτήσεις
* πώς να κρατούν μια συνομιλία
* πώς να γράφουν σε πολλές γλώσσες
* πώς να προγραμματίζουν
* κ.ά.

#### Πώς να ελέγξετε ένα μεγάλο γλωσσικό μοντέλο  
"Από όλα τα στοιχεία εισόδου σε ένα μεγάλο γλωσσικό μοντέλο, το πιο επιρροητικό κατά πολύ είναι το κείμενο προτροπής(1).

Τα μεγάλα γλωσσικά μοντέλα μπορούν να προτροπούνται να παράγουν έξοδο με μερικούς τρόπους:

Οδηγία: Πείτε στο μοντέλο τι θέλετε
Συμπλήρωση: Προκαλέστε το μοντέλο να ολοκληρώσει την αρχή αυτού που θέλετε
Επίδειξη: Δείξτε στο μοντέλο τι θέλετε, είτε με:
Λίγα παραδείγματα στην προτροπή
Εκατοντάδες ή χιλιάδες παραδείγματα σε ένα σύνολο εκπαίδευσης για λεπτομερή προσαρμογή"



#### Υπάρχουν τρεις βασικές οδηγίες για τη δημιουργία ερωτημάτων:

**Δείξτε και πείτε**. Κάντε σαφές τι θέλετε είτε μέσω οδηγιών, παραδειγμάτων, ή ενός συνδυασμού των δύο. Αν θέλετε το μοντέλο να ταξινομήσει μια λίστα αντικειμένων σε αλφαβητική σειρά ή να ταξινομήσει μια παράγραφο κατά συναίσθημα, δείξτε του ότι αυτό θέλετε.

**Παρέχετε ποιοτικά δεδομένα**. Αν προσπαθείτε να φτιάξετε έναν ταξινομητή ή να κάνετε το μοντέλο να ακολουθήσει ένα μοτίβο, βεβαιωθείτε ότι υπάρχουν αρκετά παραδείγματα. Φροντίστε να διορθώσετε τα παραδείγματά σας — το μοντέλο είναι συνήθως αρκετά έξυπνο για να διακρίνει βασικά ορθογραφικά λάθη και να σας δώσει απάντηση, αλλά μπορεί επίσης να υποθέσει ότι αυτό είναι σκόπιμο και να επηρεάσει την απάντηση.

**Ελέγξτε τις ρυθμίσεις σας.** Οι ρυθμίσεις temperature και top_p ελέγχουν πόσο καθοριστικό είναι το μοντέλο στην παραγωγή μιας απάντησης. Αν ζητάτε μια απάντηση όπου υπάρχει μόνο μία σωστή απάντηση, τότε θέλετε να τις ρυθμίσετε χαμηλότερα. Αν ψάχνετε για πιο ποικίλες απαντήσεις, τότε ίσως θέλετε να τις ρυθμίσετε ψηλότερα. Το μεγαλύτερο λάθος που κάνουν οι άνθρωποι με αυτές τις ρυθμίσεις είναι να υποθέτουν ότι ελέγχουν την «εξυπνάδα» ή τη «δημιουργικότητα».


Πηγή: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Υποβολή!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Επαναλάβετε την ίδια κλήση, πώς συγκρίνονται τα αποτελέσματα;


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Περίληψη Κειμένου  
#### Πρόκληση  
Περίληψη κειμένου προσθέτοντας ένα 'tl;dr:' στο τέλος ενός αποσπάσματος κειμένου. Παρατηρήστε πώς το μοντέλο κατανοεί πώς να εκτελεί έναν αριθμό εργασιών χωρίς επιπλέον οδηγίες. Μπορείτε να πειραματιστείτε με πιο περιγραφικά ερεθίσματα από το tl;dr για να τροποποιήσετε τη συμπεριφορά του μοντέλου και να προσαρμόσετε την περίληψη που λαμβάνετε(3).  

Πρόσφατες εργασίες έχουν δείξει σημαντικά κέρδη σε πολλές εργασίες και benchmarks φυσικής γλώσσας προ-εκπαιδεύοντας σε ένα μεγάλο σώμα κειμένων και στη συνέχεια εκτελώντας fine-tuning σε μια συγκεκριμένη εργασία. Αν και συνήθως η αρχιτεκτονική είναι ανεξάρτητη από την εργασία, αυτή η μέθοδος απαιτεί ακόμα σύνολα δεδομένων fine-tuning ειδικά για την εργασία που αποτελούνται από χιλιάδες ή δεκάδες χιλιάδες παραδείγματα. Αντίθετα, οι άνθρωποι μπορούν γενικά να εκτελέσουν μια νέα γλωσσική εργασία μόνο από λίγα παραδείγματα ή από απλές οδηγίες - κάτι που τα τρέχοντα συστήματα NLP δυσκολεύονται ακόμα να κάνουν. Εδώ δείχνουμε ότι η κλιμάκωση των γλωσσικών μοντέλων βελτιώνει σημαντικά την απόδοση σε λίγα παραδείγματα χωρίς εξειδίκευση, φτάνοντας κάποιες φορές σε ανταγωνιστικότητα με προηγούμενες μεθόδους fine-tuning.  



Tl;dr


# Ασκήσεις για διάφορες περιπτώσεις χρήσης  
1. Περίληψη Κειμένου  
2. Κατηγοριοποίηση Κειμένου  
3. Δημιουργία Νέων Ονομάτων Προϊόντων


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Κατηγοριοποίηση Κειμένου  
#### Πρόκληση  
Κατηγοριοποιήστε αντικείμενα σε κατηγορίες που παρέχονται κατά τον χρόνο συμπερασμού. Στο παρακάτω παράδειγμα, παρέχουμε τόσο τις κατηγορίες όσο και το κείμενο για κατηγοριοποίηση στο prompt(*playground_reference). 

Ερώτημα Πελάτη: Γεια σας, πρόσφατα έσπασε ένα από τα πλήκτρα στο πληκτρολόγιο του φορητού μου και θα χρειαστώ αντικατάσταση:

Κατηγοριοποιημένη κατηγορία:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Δημιουργία Νέων Ονομάτων Προϊόντων
#### Πρόκληση
Δημιουργήστε ονόματα προϊόντων από δείγματα λέξεων. Εδώ συμπεριλαμβάνουμε στην υπόδειξη πληροφορίες σχετικά με το προϊόν για το οποίο πρόκειται να δημιουργήσουμε ονόματα. Παρέχουμε επίσης ένα παρόμοιο παράδειγμα για να δείξουμε το πρότυπο που θέλουμε να λάβουμε. Έχουμε επίσης ορίσει την τιμή θερμοκρασίας υψηλά για να αυξήσουμε την τυχαιότητα και τις πιο καινοτόμες απαντήσεις.

Περιγραφή προϊόντος: Μηχανή για σπιτικά milkshake
Λέξεις-σπόροι: γρήγορο, υγιεινό, συμπαγές.
Ονόματα προϊόντων: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Περιγραφή προϊόντος: Ένα ζευγάρι παπούτσια που μπορεί να προσαρμοστεί σε οποιοδήποτε μέγεθος ποδιού.
Λέξεις-σπόροι: προσαρμόσιμο, ταιριαστό, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Αναφορές  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Βέλτιστες πρακτικές για την εκπαίδευση μοντέλων GPT για ταξινόμηση κειμένου](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Για Περισσότερη Βοήθεια  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Συνεισφέροντες
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
